# 151 — Residual Stream Bottleneck Analysis

The residual stream has fixed dimensionality (d_model), but not all dimensions
are always used. This module finds information bottlenecks: where dimensions
are underutilized, where compression occurs, and which dimensions matter most.

## Why This Matters

Residual stream bottleneck characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_bottleneck import (
    dimension_utilization,
    compression_points,
    redundancy_detection,
    capacity_allocation,
    critical_dimensions,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)

key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.arange(15)

## Dimension Utilization

How many of the d_model dimensions carry meaningful variation? Low utilization
means the model wastes capacity.

In [ ]:
result = dimension_utilization(model, tokens)

print(f"Mean utilization: {result['mean_utilization']:.2%}\n")
for p in result['per_layer']:
    bar = '#' * int(p['utilization'] * 30)
    print(f"Layer {p['layer']}: {p['active_dimensions']}/{p['total_dimensions']} "
          f"({p['utilization']:.2%}) {bar}")

## Compression Points

Find layers where the effective rank drops — these are information bottlenecks
where the representation compresses.

In [ ]:
result = compression_points(model, tokens)

print(f"Bottleneck layers: {result['bottleneck_layers']}\n")
for p in result['per_layer']:
    marker = ' <-- BOTTLENECK' if p['is_bottleneck'] else ''
    print(f"Layer {p['layer']}: effective_rank={p['effective_rank']:.1f}  "
          f"compression_ratio={p['compression_ratio']:.3f}{marker}")

## Redundancy Detection

Find dimensions that carry highly correlated information. Redundant dimensions
represent wasted capacity.

In [ ]:
result = redundancy_detection(model, tokens)

print(f"Total redundant pairs: {result['total_redundant_pairs']}\n")
for p in result['per_layer']:
    print(f"Layer {p['layer']}: {p['n_redundant_pairs']} redundant pairs, "
          f"max_corr={p['max_correlation']:.3f}, mean_abs={p['mean_abs_correlation']:.3f}")

## Capacity Allocation

How much of the final residual's variance comes from each component?
The fractions don't sum to 1 because of interference (cross-terms).

In [ ]:
result = capacity_allocation(model, tokens)

print(f"Total norm^2: {result['total_norm_squared']:.4f}")
print(f"Interference: {result['interference']:.4f}\n")

for c in result['components']:
    bar = '#' * int(c['fraction'] * 50)
    print(f"{c['component']:12s}: norm^2={c['norm_squared']:.4f}  "
          f"fraction={c['fraction']:.3f}  {bar}")

## Critical Dimensions

Which dimensions of the residual stream matter most for the final prediction?
Decomposes the logit into per-dimension contributions via the unembedding.

In [ ]:
result = critical_dimensions(model, tokens, top_k=10)

print(f"Target token: {result['target_token']}")
print(f"Top-10 fraction of total logit: {result['top_k_fraction']:.2%}\n")

for d in result['per_dimension']:
    direction = '+' if d['logit_contribution'] > 0 else '-'
    bar = '#' * int(abs(d['logit_contribution']) * 20)
    print(f"dim {d['dimension']:3d}: resid={d['residual_value']:+.4f} * "
          f"unembed={d['unembed_weight']:+.4f} = {d['logit_contribution']:+.4f} {direction}{bar}")